# Phishing Website Detection — 06: Additional Corpora and Statistical Inference

Two goals, and they are not the same goal.

**More corpora.** The current analysis rests on two datasets. That is thin for a
claim about benchmark validity, and it is the genuine weakness of the paper. This
notebook adds every publicly available corpus that ships raw URLs and can be
retrieved programmatically.

**Proper inference.** Cross-dataset results currently use a single seed and no
uncertainty estimate. We add stratified bootstrap confidence intervals, the DeLong
test for paired AUC comparisons, Holm--Bonferroni correction across the transfer
matrix, and a permutation test on the within-versus-cross gap.

A note on the framing. Statistical significance is not something a study can be
given more of by wishing; it follows from effect size and sample size. The effect
here is already very large --- roughly 0.999 against 0.608 ROC-AUC --- so
significance was never the vulnerable part. What the paper lacks is *breadth*:
one documented artefact in one corpus is an anecdote, and several corpora audited
with the same instrument is a survey. That reframing is the real reason to add
datasets.

### What changes about the paper

Section 3 currently says "PhiUSIIL contains a structural artefact." With more
corpora it can say "we audited $N$ public phishing URL corpora with a common
structural diagnostic; $k$ of them show class-dependent structural signatures."
That is a considerably stronger contribution and it is what Cell 5 produces.

### Fixes in this version

`urlparse` is lazy: it accepts a malformed string and raises only when you touch
`.hostname` or `.port`. Real abuse feeds carry control characters in the authority
section, which killed Cell 6 with `Port could not be cast to integer value`. The
module written in Cell 1b now guards every raising access and records a `malformed`
flag, so the rate becomes a reportable data-quality fact instead of a crash.

Cell 5b is new. Before treating $N$ corpora as $N$ observations it checks they are
$N$ different datasets --- in the first run, `hannousse` and `kaggle_linkphish`
agreed on five structural properties to within 0.2 percentage points, which is not
something independent collections do.

### Prior work you must cite

Mia, Derakhshan and Pritom (NSysS 2024, arXiv:2411.09813) report cross-dataset
accuracy dropping to 51 and 59 per cent between two phishing corpora and conclude
that features are dataset-dependent. Position your contribution against theirs
explicitly: they observed degradation, you identify a mechanism; they did not use
PhiUSIIL; their degradation was roughly symmetric whereas yours is not, which is
what distinguishes "features do not generalise" from "one corpus is defective".
Their recommendation to merge datasets for training also deserves comment, since
merging propagates a structural artefact rather than removing it.

## Cell 1 — Environment

In [1]:
!pip install -q ucimlrepo tldextract scikit-learn pandas numpy matplotlib scipy xgboost lightgbm kagglehub

from google.colab import drive
from pathlib import Path
import sys, os, json, warnings
warnings.filterwarnings("ignore")

drive.mount('/content/drive')

PROJECT   = "phishing-detection"
REPO_DIR  = Path("/content") / PROJECT
DRIVE_DIR = Path("/content/drive/MyDrive/research") / PROJECT

for d in ["data/raw", "data/external", "checkpoints", "models", "logs", "notebooks"]:
    (DRIVE_DIR / d).mkdir(parents=True, exist_ok=True)
for d in ["src", "src/data", "src/features", "src/evaluation", "src/utils",
          "reports", "reports/tables", "reports/figures"]:
    (REPO_DIR / d).mkdir(parents=True, exist_ok=True)
for d in ["src", "src/data", "src/features", "src/evaluation", "src/utils"]:
    (REPO_DIR / d / "__init__.py").touch()

for name in ["data", "checkpoints", "models", "logs"]:
    link = REPO_DIR / name
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        raise RuntimeError(f"{link} exists and is not a symlink")
    link.symlink_to(DRIVE_DIR / name)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
RAW = REPO_DIR / "data" / "raw"
print("raw ->", RAW.resolve())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 2.8 MB/s eta 0:00:00
Mounted at /content/drive
raw -> /content/drive/MyDrive/research/phishing-detection/data/raw


## Cell 1b — Restore modules

In [2]:
CKPT_SRC = 'import json, time, hashlib, subprocess\nfrom pathlib import Path\nimport joblib\n\n\ndef _git_hash(repo="/content/phishing-detection"):\n    try:\n        return subprocess.check_output(\n            ["git", "-C", repo, "rev-parse", "--short", "HEAD"],\n            stderr=subprocess.DEVNULL).decode().strip()\n    except Exception:\n        return "nogit"\n\n\nclass Checkpoint:\n    # Resumable checkpointing for Colab. Survives runtime resets via Drive.\n\n    def __init__(self, root="/content/phishing-detection/checkpoints",\n                 run_id=None, verbose=True):\n        self.root = Path(root)\n        self.run_id = run_id or time.strftime("run_%Y%m%d_%H%M%S")\n        self.dir = self.root / self.run_id\n        self.dir.mkdir(parents=True, exist_ok=True)\n        self.verbose = verbose\n        self.manifest_path = self.dir / "manifest.json"\n        self.manifest = (json.loads(self.manifest_path.read_text())\n                         if self.manifest_path.exists() else {})\n\n    @staticmethod\n    def _key(params):\n        if params is None:\n            return "default"\n        blob = json.dumps(params, sort_keys=True, default=str)\n        return hashlib.md5(blob.encode()).hexdigest()[:10]\n\n    def _path(self, stage, params=None):\n        return self.dir / (stage + "__" + self._key(params) + ".joblib")\n\n    def exists(self, stage, params=None):\n        return self._path(stage, params).exists()\n\n    def save(self, stage, obj, params=None, meta=None):\n        p = self._path(stage, params)\n        joblib.dump(obj, p, compress=3)\n        size_mb = round(p.stat().st_size / 1e6, 3)\n        self.manifest[p.name] = {\n            "stage": stage, "params": params, "meta": meta or {},\n            "git_commit": _git_hash(),\n            "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n            "size_mb": size_mb,\n        }\n        self.manifest_path.write_text(\n            json.dumps(self.manifest, indent=2, default=str))\n        if self.verbose:\n            print("[ckpt] saved " + p.name + "  (" + str(size_mb) + " MB)")\n        return p\n\n    def load(self, stage, params=None):\n        p = self._path(stage, params)\n        if not p.exists():\n            raise FileNotFoundError("No checkpoint: " + str(p))\n        if self.verbose:\n            print("[ckpt] loaded " + p.name)\n        return joblib.load(p)\n\n    def cache(self, stage, fn, params=None, meta=None, force=False):\n        if self.exists(stage, params) and not force:\n            return self.load(stage, params)\n        if self.verbose:\n            print("[ckpt] computing " + stage + " ...")\n        out = fn()\n        self.save(stage, out, params=params, meta=meta)\n        return out\n\n    def summary(self):\n        import pandas as pd\n        if not self.manifest:\n            return pd.DataFrame()\n        return pd.DataFrame(self.manifest).T[\n            ["stage", "saved_at", "size_mb", "git_commit"]]\n\n    @classmethod\n    def latest_run(cls, root="/content/phishing-detection/checkpoints", **kw):\n        runs = sorted(p.name for p in Path(root).glob("run_*") if p.is_dir())\n        if not runs:\n            raise FileNotFoundError("No previous runs.")\n        return cls(root=root, run_id=runs[-1], **kw)\n'

URLFEAT_SRC = '"""Canonical URL-lexical features + split strategies.\n\nHardened against malformed URLs. Real corpora contain entries that make\nurllib.parse raise: non-numeric or out-of-range ports, unbalanced square\nbrackets that look like broken IPv6 literals, and embedded control\ncharacters. urlparse() is lazy about this - it raises on the .hostname\nand .port properties rather than at parse time - so every access is\nguarded here.\n"""\nimport re, math\nfrom urllib.parse import urlparse, SplitResult\nimport numpy as np\nimport pandas as pd\n\nSUSPICIOUS_WORDS = [\n    "login", "signin", "verify", "account", "update", "secure", "webscr",\n    "banking", "confirm", "password", "credential", "invoice", "payment",\n    "billing", "suspend", "unlock", "recover", "wallet", "support",\n]\nSHORTENERS = {\n    "bit.ly", "goo.gl", "tinyurl.com", "t.co", "ow.ly", "is.gd", "buff.ly",\n    "adf.ly", "bit.do", "cutt.ly", "rb.gy", "shorte.st", "rebrand.ly",\n}\nIPV4 = re.compile(r"^(?:\\d{1,3}\\.){3}\\d{1,3}$")\nHEX_HOST = re.compile(r"^0x[0-9a-f]+$", re.I)\nCTRL = re.compile(r"[\\x00-\\x1f\\x7f]")\n\n_EMPTY = SplitResult(scheme="", netloc="", path="", query="", fragment="")\n\n\ndef safe_parse(url):\n    """Parse a URL, never raising. Returns a SplitResult-like object.\n\n    Falls back through three stages: parse as given; strip control\n    characters and square brackets and retry; return an empty result.\n    """\n    s = str(url).strip()\n    if "://" not in s:\n        s = "http://" + s\n    try:\n        p = urlparse(s)\n        _ = p.netloc          # force any lazy validation we can\n        return p\n    except ValueError:\n        pass\n    cleaned = CTRL.sub("", s).replace("[", "").replace("]", "")\n    try:\n        return urlparse(cleaned)\n    except ValueError:\n        return _EMPTY\n\n\ndef safe_hostname(p):\n    """netloc-derived hostname, tolerating malformed authority sections."""\n    try:\n        h = p.hostname\n        if h is not None:\n            return h.lower()\n    except ValueError:\n        pass\n    netloc = getattr(p, "netloc", "") or ""\n    netloc = netloc.rsplit("@", 1)[-1]          # drop userinfo\n    netloc = CTRL.sub("", netloc).replace("[", "").replace("]", "")\n    host = netloc.split(":", 1)[0]\n    return host.lower()\n\n\ndef safe_port(p):\n    """Port if it is a valid integer in range, else None. Never raises."""\n    try:\n        return p.port\n    except ValueError:\n        return None\n\n\ndef is_malformed(url):\n    """True if the URL needed the fallback path. Report this rate."""\n    s = str(url).strip()\n    if "://" not in s:\n        s = "http://" + s\n    try:\n        p = urlparse(s)\n        _ = p.hostname\n        _ = p.port\n        return False\n    except ValueError:\n        return True\n\n\ndef _entropy(s):\n    if not s:\n        return 0.0\n    counts = {}\n    for ch in s:\n        counts[ch] = counts.get(ch, 0) + 1\n    n = len(s)\n    return -sum((c / n) * math.log2(c / n) for c in counts.values())\n\n\ndef canonical_url_features(url, extractor=None):\n    """~32 lexical features computable from a URL alone, identically for any source."""\n    url = str(url).strip()\n    if "://" not in url:\n        url = "http://" + url\n\n    p = safe_parse(url)\n    host = safe_hostname(p)\n    path = getattr(p, "path", "") or ""\n    query = getattr(p, "query", "") or ""\n\n    subdomain = domain = suffix = ""\n    if extractor is not None:\n        try:\n            ext = extractor(url)\n            subdomain, domain, suffix = ext.subdomain, ext.domain, ext.suffix\n        except Exception:\n            extractor = None\n    if not (domain or suffix):\n        parts = [x for x in host.split(".") if x]\n        suffix = parts[-1] if len(parts) > 1 else ""\n        domain = parts[-2] if len(parts) > 2 else (parts[0] if parts else "")\n        subdomain = ".".join(parts[:-2]) if len(parts) > 2 else ""\n\n    registrable = f"{domain}.{suffix}".strip(".")\n    digits = sum(ch.isdigit() for ch in url)\n    letters = sum(ch.isalpha() for ch in url)\n\n    return {\n        "url_length": len(url),\n        "host_length": len(host),\n        "path_length": len(path),\n        "query_length": len(query),\n        "n_dots": url.count("."),\n        "n_hyphens": url.count("-"),\n        "n_underscores": url.count("_"),\n        "n_slashes": path.count("/"),\n        "n_question": url.count("?"),\n        "n_equals": url.count("="),\n        "n_at": url.count("@"),\n        "n_ampersand": url.count("&"),\n        "n_percent": url.count("%"),\n        "n_digits": digits,\n        "n_letters": letters,\n        "digit_ratio": digits / max(len(url), 1),\n        "n_subdomains": len([s for s in subdomain.split(".") if s]),\n        "subdomain_length": len(subdomain),\n        "domain_length": len(domain),\n        "tld_length": len(suffix),\n        "is_ip_host": int(bool(IPV4.match(host))),\n        "is_hex_host": int(bool(HEX_HOST.match(host))),\n        "is_https": int(str(getattr(p, "scheme", "")).lower() == "https"),\n        "has_port": int(safe_port(p) is not None),\n        "is_shortener": int(registrable in SHORTENERS),\n        "has_punycode": int("xn--" in host),\n        "has_double_slash_path": int("//" in path),\n        "n_suspicious_words": sum(w in url.lower() for w in SUSPICIOUS_WORDS),\n        "url_entropy": _entropy(url),\n        "host_entropy": _entropy(host),\n        "longest_token_path": max([len(t) for t in re.split(r"[/\\-_.]", path) if t] or [0]),\n        "has_https_token_in_host": int("https" in host),\n    }\n\n\ndef build_canonical_matrix(urls, use_tldextract=True):\n    extractor = None\n    if use_tldextract:\n        extractor = get_extractor()\n    rows = [canonical_url_features(u, extractor) for u in urls]\n    return pd.DataFrame(rows, index=getattr(urls, "index", None))\n\n\n_DEFAULT_EXTRACTOR = None\n\n\ndef get_extractor():\n    """Cached tldextract instance using the bundled PSL snapshot (no network)."""\n    global _DEFAULT_EXTRACTOR\n    if _DEFAULT_EXTRACTOR is None:\n        try:\n            import tldextract\n            _DEFAULT_EXTRACTOR = tldextract.TLDExtract(suffix_list_urls=())\n        except Exception:\n            _DEFAULT_EXTRACTOR = False\n    return _DEFAULT_EXTRACTOR or None\n\n\ndef registrable_domain(url, extractor="auto"):\n    """eTLD+1. Returns the bare IP for IP-literal hosts. Never raises."""\n    s = str(url).strip()\n    if "://" not in s:\n        s = "http://" + s\n    host = safe_hostname(safe_parse(s))\n    if not host:\n        return ""\n    if IPV4.match(host):\n        return host\n    if extractor == "auto":\n        extractor = get_extractor()\n    if extractor is not None:\n        try:\n            e = extractor(s)\n            reg = f"{e.domain}.{e.suffix}".strip(".").strip("[]")\n            if reg:\n                return reg\n        except Exception:\n            pass\n    parts = [x for x in host.strip("[]").split(".") if x]\n    return ".".join(parts[-2:]) if len(parts) >= 2 else host.strip("[]")\n\n\n# ------------------------------------------------------------------ splits\ndef grouped_split(X, y, groups, test_size=0.2, seed=42):\n    """No registrable domain appears in both train and test."""\n    from sklearn.model_selection import GroupShuffleSplit\n    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)\n    tr, te = next(gss.split(X, y, groups))\n    return tr, te\n\n\ndef stratified_split(X, y, test_size=0.2, seed=42):\n    from sklearn.model_selection import train_test_split\n    idx = np.arange(len(y))\n    return train_test_split(idx, test_size=test_size, random_state=seed, stratify=y)\n\n\ndef rebalance_to_base_rate(y, target_rate, seed=42, index=None):\n    """Downsample the phishing class to a realistic prevalence."""\n    rng = np.random.default_rng(seed)\n    idx = np.arange(len(y)) if index is None else np.asarray(index)\n    pos = idx[np.asarray(y) == 1]\n    neg = idx[np.asarray(y) == 0]\n    n_pos = int(round(target_rate * len(neg) / (1 - target_rate)))\n    n_pos = min(n_pos, len(pos))\n    keep = np.concatenate([neg, rng.choice(pos, size=n_pos, replace=False)])\n    rng.shuffle(keep)\n    return keep\n'

(REPO_DIR / "src" / "utils" / "checkpoint.py").write_text(CKPT_SRC)
(REPO_DIR / "src" / "features" / "urlfeat.py").write_text(URLFEAT_SRC)
import importlib
import src.utils.checkpoint, src.features.urlfeat
importlib.reload(src.utils.checkpoint); importlib.reload(src.features.urlfeat)
from src.utils.checkpoint import Checkpoint
import src.features.urlfeat as uf
print("modules restored")

modules restored


## Cell 2 — Statistics module

Writes `src/evaluation/stats.py`. DeLong is implemented from the fast midrank
algorithm and was verified against `sklearn.roc_auc_score` for the point estimate
and against a stratified bootstrap for the interval.

In [3]:
%%writefile /content/phishing-detection/src/evaluation/stats.py
"""Inference for transfer experiments: DeLong, bootstrap CIs, corrections."""
import numpy as np
import pandas as pd
from scipy import stats


# ------------------------------------------------------------------ DeLong
def _midrank(x):
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T
    return T2


def _structural_components(scores, n_pos, n_neg):
    """V10 (positives) and V01 (negatives) components for each predictor."""
    k = scores.shape[0]
    tx = np.empty([k, n_pos]); ty = np.empty([k, n_neg]); tz = np.empty([k, n_pos + n_neg])
    for r in range(k):
        pos = scores[r, :n_pos]
        neg = scores[r, n_pos:]
        tx[r, :] = _midrank(pos)
        ty[r, :] = _midrank(neg)
        tz[r, :] = _midrank(scores[r, :])
    aucs = (tz[:, :n_pos].sum(axis=1) / n_pos - (n_pos + 1) / 2) / n_neg
    v01 = (tz[:, :n_pos] - tx) / n_neg
    v10 = 1 - (tz[:, n_pos:] - ty) / n_pos
    return aucs, v01, v10


def delong_test(y_true, scores_a, scores_b):
    """Paired DeLong test for two AUCs on the same sample.

    Returns (auc_a, auc_b, z, p). Two-sided.
    """
    y = np.asarray(y_true).astype(int)
    order = np.argsort(-y, kind="mergesort")   # positives first
    y = y[order]
    a = np.asarray(scores_a)[order]
    b = np.asarray(scores_b)[order]
    n_pos = int(y.sum()); n_neg = len(y) - n_pos
    if n_pos == 0 or n_neg == 0:
        return np.nan, np.nan, np.nan, np.nan

    scores = np.vstack([a, b])
    aucs, v01, v10 = _structural_components(scores, n_pos, n_neg)
    s01 = np.cov(v01); s10 = np.cov(v10)
    s = s01 / n_pos + s10 / n_neg
    var = s[0, 0] + s[1, 1] - 2 * s[0, 1]
    if var <= 0:
        return float(aucs[0]), float(aucs[1]), np.nan, np.nan
    z = (aucs[0] - aucs[1]) / np.sqrt(var)
    p = 2 * stats.norm.sf(abs(z))
    return float(aucs[0]), float(aucs[1]), float(z), float(p)


def delong_ci(y_true, scores, alpha=0.05):
    """DeLong analytic confidence interval for a single AUC."""
    y = np.asarray(y_true).astype(int)
    order = np.argsort(-y, kind="mergesort")
    y = y[order]; s = np.asarray(scores)[order]
    n_pos = int(y.sum()); n_neg = len(y) - n_pos
    if n_pos == 0 or n_neg == 0:
        return np.nan, np.nan, np.nan
    aucs, v01, v10 = _structural_components(s.reshape(1, -1), n_pos, n_neg)
    var = np.var(v01, ddof=1) / n_pos + np.var(v10, ddof=1) / n_neg
    se = np.sqrt(var)
    zc = stats.norm.ppf(1 - alpha / 2)
    auc = float(aucs[0])
    return auc, max(0.0, auc - zc * se), min(1.0, auc + zc * se)


# --------------------------------------------------------------- bootstrap
def bootstrap_ci(y_true, scores, metric_fn, n_boot=2000, alpha=0.05, seed=42):
    """Stratified percentile bootstrap CI for any metric(y, s)."""
    rng = np.random.default_rng(seed)
    y = np.asarray(y_true); s = np.asarray(scores)
    pos = np.flatnonzero(y == 1); neg = np.flatnonzero(y == 0)
    if len(pos) < 2 or len(neg) < 2:
        return np.nan, np.nan, np.nan
    vals = np.empty(n_boot)
    for i in range(n_boot):
        idx = np.concatenate([rng.choice(pos, len(pos), replace=True),
                              rng.choice(neg, len(neg), replace=True)])
        try:
            vals[i] = metric_fn(y[idx], s[idx])
        except Exception:
            vals[i] = np.nan
    vals = vals[~np.isnan(vals)]
    point = metric_fn(y, s)
    lo, hi = np.percentile(vals, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(point), float(lo), float(hi)


# ------------------------------------------------------- permutation tests
def permutation_gap_test(within_vals, cross_vals, n_perm=10000, seed=42):
    """Is the within-vs-cross gap larger than chance reassignment would give?"""
    rng = np.random.default_rng(seed)
    a = np.asarray(within_vals, dtype=float); b = np.asarray(cross_vals, dtype=float)
    obs = a.mean() - b.mean()
    pool = np.concatenate([a, b]); n = len(a)
    count = 0
    for _ in range(n_perm):
        rng.shuffle(pool)
        if (pool[:n].mean() - pool[n:].mean()) >= obs:
            count += 1
    return float(obs), (count + 1) / (n_perm + 1)


def cliffs_delta(a, b):
    """Non-parametric effect size. |d|: .147 small, .33 medium, .474 large."""
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    gt = sum((x > y) for x in a for y in b)
    lt = sum((x < y) for x in a for y in b)
    return (gt - lt) / (len(a) * len(b))


# --------------------------------------------- multiple comparison control
def holm_bonferroni(pvals, alpha=0.05):
    """Holm step-down. Returns (adjusted p, reject) in the original order."""
    p = np.asarray(pvals, dtype=float)
    ok = ~np.isnan(p)
    out = np.full(len(p), np.nan); rej = np.zeros(len(p), dtype=bool)
    idx = np.flatnonzero(ok)
    if len(idx) == 0:
        return out, rej
    order = idx[np.argsort(p[idx])]
    m = len(order)
    running = 0.0
    for rank, i in enumerate(order):
        adj = min(1.0, (m - rank) * p[i])
        running = max(running, adj)     # enforce monotonicity
        out[i] = running
        rej[i] = running <= alpha
    return out, rej

Writing /content/phishing-detection/src/evaluation/stats.py


## Cell 3 — Acquire additional URL-bearing corpora

Only corpora that publish the raw URL string can join the canonical feature space.
Each downloader degrades to printed instructions rather than aborting the notebook.

**Kaggle sources need credentials.** In Colab, add your `kaggle.json` username and
key as secrets named `KAGGLE_USERNAME` and `KAGGLE_KEY`, or upload `kaggle.json`
when prompted. If you skip this, those corpora are simply reported as unavailable.

In [4]:
import pandas as pd, numpy as np, io, zipfile, urllib.request, json

UA = {"User-Agent": "Mozilla/5.0 (academic research)"}
def _get(url, timeout=300):
    return urllib.request.urlopen(urllib.request.Request(url, headers=UA),
                                  timeout=timeout).read()

NEW = {}   # key -> DataFrame with columns ['url','label']  (label 1 = phishing)

def _save(key, df):
    out = RAW / key; out.mkdir(parents=True, exist_ok=True)
    df.to_csv(out / f"{key}_urls.csv", index=False)
    NEW[key] = df
    print(f"[{key}] {len(df):,} rows  phishing_rate={df['label'].mean():.4f}")

# ---------------------------------------------------------------- Kaggle 1
def get_kaggle_malicious():
    key = "kaggle_malicious"
    p = RAW / key / f"{key}_urls.csv"
    if p.exists():
        _save(key, pd.read_csv(p)); return
    try:
        import kagglehub
        path = kagglehub.dataset_download("sid321axn/malicious-urls-dataset")
        f = next(Path(path).rglob("*.csv"))
        d = pd.read_csv(f)
        ucol = [c for c in d.columns if c.lower() == "url"][0]
        tcol = [c for c in d.columns if c.lower() in ("type", "label", "class")][0]
        t = d[tcol].astype(str).str.lower()
        keep = t.isin(["benign", "phishing"])
        out = pd.DataFrame({"url": d.loc[keep, ucol].astype(str),
                            "label": (t[keep] == "phishing").astype(int)})
        _save(key, out.reset_index(drop=True))
    except Exception as e:
        print(f"[{key}] unavailable: {type(e).__name__}: {e}")
        print("   manual: kaggle.com/datasets/sid321axn/malicious-urls-dataset")

# ---------------------------------------------------------------- Kaggle 2
def get_kaggle_linkphish():
    """The corpus used as D2 by Mia et al. (2024) - enables direct comparison."""
    key = "kaggle_linkphish"
    p = RAW / key / f"{key}_urls.csv"
    if p.exists():
        _save(key, pd.read_csv(p)); return
    try:
        import kagglehub
        path = kagglehub.dataset_download("winson13/dataset-for-link-phishing-detection")
        cand = [f for f in Path(path).rglob("*.csv")]
        for f in cand:
            d = pd.read_csv(f, nrows=5)
            ucol = [col for col in d.columns if col.lower() in ("url", "urls")]
            if ucol:
                d = pd.read_csv(f, low_memory=False)
                lcol = [col for col in d.columns
                        if col.lower() in ("status", "label", "class", "phishing")][0]
                s = d[lcol]
                lab = (s.astype(str).str.lower().isin(["phishing", "1", "true"])
                       if s.dtype == object else (s == 1)).astype(int)
                _save(key, pd.DataFrame({"url": d[ucol[0]].astype(str), "label": lab}))
                return
        print(f"[{key}] no URL column found in download")
    except Exception as e:
        print(f"[{key}] unavailable: {type(e).__name__}: {e}")

# -------------------------------------------------------------- Mendeley 2
def get_mendeley_phishurl():
    key = "mendeley_phishurl"
    p = RAW / key / f"{key}_urls.csv"
    if p.exists():
        _save(key, pd.read_csv(p)); return
    out = RAW / key; out.mkdir(parents=True, exist_ok=True)
    try:
        api = ("https://data.mendeley.com/public-api/datasets/vfszbj9b36/files"
               "?folder_id=root&version=1")
        files = json.loads(_get(api, timeout=90))
        for f in files:
            name = f.get("filename", "")
            link = (f.get("content_details") or {}).get("download_url")
            if link:
                (out / name).write_bytes(_get(link))
                print(f"   got {name}")
        frames = []
        for f in out.glob("*.csv"):
            d = pd.read_csv(f, low_memory=False)
            uc = [c for c in d.columns if "url" in c.lower()]
            if not uc:
                continue
            lc = [c for c in d.columns
                  if c.lower() in ("label", "status", "type", "class", "phishing")]
            if lc:
                s = d[lc[0]]
                lab = (s.astype(str).str.lower().isin(["phishing", "malicious", "1"])
                       if s.dtype == object else (s == 1)).astype(int)
            else:
                lab = pd.Series(1 if "phish" in f.name.lower() else 0, index=d.index)
            frames.append(pd.DataFrame({"url": d[uc[0]].astype(str), "label": lab}))
        if frames:
            _save(key, pd.concat(frames, ignore_index=True))
        else:
            print(f"[{key}] no usable URL column")
    except Exception as e:
        print(f"[{key}] unavailable: {type(e).__name__}: {e}")
        print("   manual: data.mendeley.com/datasets/vfszbj9b36/1")

# ------------------------------------------------------------- Tan (retry)
def get_tan():
    key = "tan"
    try:
        api = ("https://data.mendeley.com/public-api/datasets/h3cgnj8hft/files"
               "?folder_id=root&version=1")
        files = json.loads(_get(api, timeout=90))
        out = RAW / key; out.mkdir(parents=True, exist_ok=True)
        for f in files:
            link = (f.get("content_details") or {}).get("download_url")
            if link:
                (out / f["filename"]).write_bytes(_get(link))
        cands = list(out.rglob("*.csv"))
        if cands:
            d = pd.read_csv(max(cands, key=lambda x: x.stat().st_size), low_memory=False)
            uc = [c for c in d.columns if "url" in c.lower() and d[c].dtype == object]
            print(f"[{key}] downloaded, shape={d.shape}, "
                  f"raw-URL column: {uc[0] if uc else 'NONE (feature matrix only)'}")
        else:
            print(f"[{key}] no CSV found")
    except Exception as e:
        print(f"[{key}] unavailable: {type(e).__name__}: {e}")

for fn in [get_kaggle_malicious, get_kaggle_linkphish,
           get_mendeley_phishurl, get_tan]:
    fn()

print(f"\n{len(NEW)} new URL-bearing corpora acquired")

[kaggle_malicious] 522,214 rows  phishing_rate=0.1802
[kaggle_linkphish] 19,431 rows  phishing_rate=0.5000
[mendeley_phishurl] 504,983 rows  phishing_rate=0.3153
[tan] no CSV found

3 new URL-bearing corpora acquired


### A corpus we deliberately do not build

JPCERT/CC publishes a dated phishing URL list on GitHub, and pairing it with
Tranco would yield a large, recent, timestamped corpus. We do not do this, and the
reason matters for the paper's credibility.

Tranco supplies domain roots; an abuse feed supplies full URLs. Combining them
reproduces exactly the structural mismatch this paper identifies in PhiUSIIL. It
would be inconsistent to criticise that construction and then rely on it. If you
want a temporal corpus, the benign side has to be crawled to full URLs at
comparable depth — which is worth doing, and is a separate piece of work.

## Cell 4 — Load existing corpora and unify

Existing URL-bearing corpora from earlier notebooks are added to the same
structure so every dataset is treated identically from here on.

In [5]:
EXISTING = {
    "phiusiil":  ("phiusiil/phiusiil.csv",   "URL",  "label",  0),
    "hannousse": ("hannousse/hannousse.csv", "url",  "status", "phishing"),
}

CORPORA = {}
for key, (rel, ucol, lcol, phish) in EXISTING.items():
    p = RAW / rel
    if not p.exists():
        print(f"[{key}] missing"); continue
    d = pd.read_csv(p, low_memory=False)
    uc = [c for c in d.columns if c.lower() == ucol.lower()][0]
    lc = [c for c in d.columns if c.lower() == lcol.lower()][0]
    lab = ((d[lc].astype(str).str.lower() == str(phish).lower()).astype(int)
           if isinstance(phish, str) else (d[lc] == phish).astype(int))
    CORPORA[key] = pd.DataFrame({"url": d[uc].astype(str), "label": lab})

for key, df in NEW.items():
    CORPORA[key] = df

# Cap very large corpora so the transfer matrix stays tractable
CAP = 60000
rng = np.random.default_rng(42)
for key, df in list(CORPORA.items()):
    if len(df) > CAP:
        idx = np.sort(rng.choice(len(df), CAP, replace=False))
        CORPORA[key] = df.iloc[idx].reset_index(drop=True)

summary = pd.DataFrame([
    {"corpus": k, "n": len(v), "phishing_rate": round(float(v["label"].mean()), 4)}
    for k, v in CORPORA.items()]).sort_values("n", ascending=False)
print(summary.to_string(index=False))
print(f"\n{len(CORPORA)} corpora in the transfer experiment")

           corpus     n  phishing_rate
         phiusiil 60000         0.4267
 kaggle_malicious 60000         0.1805
mendeley_phishurl 60000         0.3155
 kaggle_linkphish 19431         0.5000
        hannousse 11430         0.5000

5 corpora in the transfer experiment


## Cell 5 — Structural audit across all corpora

The diagnostic that found the PhiUSIIL artefact, applied uniformly. For each
corpus and class we record the share of URLs using HTTPS, the share with an empty
path, and the share carrying a query string.

**What to look for.** A value of exactly 0 or exactly 100 per cent within one
class is a structural signature: the collection procedure, not the phenomenon, is
determining that property. The `max_gap` column gives the largest between-class
difference across the three properties, which is a single-number severity score.

Real corpora contain URLs that `urlparse` rejects outright, usually because of
stray square brackets that look like a malformed IPv6 literal. These are parsed
through a bracket-stripping fallback rather than dropped, and the `malformed_pct`
column records how many there were. That rate is worth reporting: it is a
data-quality property of the corpus, and a corpus with a high rate has had less
curation than its documentation implies.

In [6]:
from urllib.parse import urlparse

def structural_profile(urls):
    """Uses uf.safe_parse so that malformed URLs are profiled rather than fatal."""
    p = urls.map(uf.safe_parse)
    return pd.DataFrame({
        "https":      urls.astype(str).str.strip().str.startswith("https").astype(int),
        "empty_path": p.map(lambda x: len(getattr(x, "path", "") or "") == 0).astype(int),
        "has_query":  p.map(lambda x: len(getattr(x, "query", "") or "") > 0).astype(int),
    })

rows = []
for key, df in CORPORA.items():
    prof = structural_profile(df["url"])
    r = {"corpus": key, "n": len(df)}
    gaps = []
    for col in ["https", "empty_path", "has_query"]:
        leg = prof.loc[df["label"] == 0, col].mean()
        ph  = prof.loc[df["label"] == 1, col].mean()
        r[f"{col}_legit"] = round(100 * leg, 2)
        r[f"{col}_phish"] = round(100 * ph, 2)
        gaps.append(abs(leg - ph))
    r["max_gap_pp"] = round(100 * max(gaps), 2)
    # degenerate = a property taking a single value across an entire class
    deg = 0
    for col in ["https", "empty_path", "has_query"]:
        for lab in (0, 1):
            v = prof.loc[df["label"] == lab, col].mean()
            if v in (0.0, 1.0):
                deg += 1
    r["degenerate_cells"] = deg
    r["malformed_pct"] = round(100 * df["url"].map(uf.is_malformed).mean(), 4)
    rows.append(r)

audit = pd.DataFrame(rows).sort_values("max_gap_pp", ascending=False)
pd.set_option("display.width", 200)
print(audit.to_string(index=False))
audit.to_csv(REPO_DIR / "reports" / "tables" / "structural_audit.csv", index=False)
print("\nsaved -> reports/tables/structural_audit.csv")
print("\nCorpora with at least one degenerate class property:")
print(list(audit.loc[audit.degenerate_cells > 0, "corpus"]))

bad = audit.loc[audit.malformed_pct > 0, ["corpus", "malformed_pct"]]
if len(bad):
    print("\nCorpora containing URLs that urlparse rejects outright:")
    print(bad.to_string(index=False))
    print("These are parsed by a bracket-stripping fallback rather than dropped.")
    print("Report the rate in the paper - it is a data-quality fact about the corpus.")

           corpus     n  https_legit  https_phish  empty_path_legit  empty_path_phish  has_query_legit  has_query_phish  max_gap_pp  degenerate_cells  malformed_pct
mendeley_phishurl 60000        100.0        34.80              0.74              5.02            13.56            16.20       65.20                 1         0.0000
         phiusiil 60000        100.0        48.55            100.00             40.39             0.00             5.96       59.61                 3         0.0000
 kaggle_malicious 60000          0.5         7.68              0.07             23.40            17.12             8.41       23.32                 0         0.0067
        hannousse 11430         44.5        33.32              6.16              5.02             3.34            23.67       20.33                 0         0.0000
 kaggle_linkphish 19431         44.5        33.28              6.23              5.21             3.32            23.61       20.29                 0         0.0000

saved -> 

## Cell 5b — Corpus independence

Before treating $N$ corpora as $N$ observations, verify they are $N$ different
datasets. Compares exact URLs, normalised URLs (lowercased, scheme and trailing
slash stripped), and registrable domains.

**Read containment, not Jaccard.** Jaccard is suppressed when one corpus is much
larger, so a subset relationship can look small; containment is
$|A \cap B| / \min(|A|,|B|)$ and goes to 1 when one corpus sits inside the other.

In [7]:
def normalise(u):
    s = str(u).strip().lower()
    for pre in ("https://", "http://"):
        if s.startswith(pre):
            s = s[len(pre):]
            break
    if s.startswith("www."):
        s = s[4:]
    return s.rstrip("/")

sets_norm, sets_dom = {}, {}
for key, df in CORPORA.items():
    u = df["url"].astype(str)
    sets_norm[key] = set(u.map(normalise))
    sets_dom[key]  = set(u.map(uf.registrable_domain))

ALL = list(CORPORA)
rows = []
for i, a in enumerate(ALL):
    for b in ALL[i+1:]:
        r = {"corpus_a": a, "corpus_b": b}
        for name, S in [("norm", sets_norm), ("domain", sets_dom)]:
            inter = len(S[a] & S[b]); union = len(S[a] | S[b])
            mn = min(len(S[a]), len(S[b]))
            r[f"{name}_overlap"]     = inter
            r[f"{name}_jaccard"]     = round(inter / union, 4) if union else 0.0
            r[f"{name}_containment"] = round(inter / mn, 4) if mn else 0.0
        rows.append(r)

overlap = pd.DataFrame(rows).sort_values("norm_containment", ascending=False)
pd.set_option("display.width", 220)
print(overlap.to_string(index=False))
overlap.to_csv(REPO_DIR / "reports" / "tables" / "corpus_overlap.csv", index=False)
print("\nsaved -> reports/tables/corpus_overlap.csv")

# within-corpus duplication is the same hygiene point at smaller scale
print("\nWithin-corpus duplication:")
for key, df in CORPORA.items():
    u = df["url"].astype(str)
    print(f"  {key:20s} rows={len(u):>7,}  unique_raw={u.nunique():>7,}  "
          f"unique_normalised={len(sets_norm[key]):>7,}")

flagged = overlap[overlap["norm_containment"] > 0.5]
if len(flagged):
    print("\nNOT INDEPENDENT (normalised containment > 0.5):")
    for _, r in flagged.iterrows():
        print(f"  {r.corpus_a} / {r.corpus_b}: {r.norm_overlap:,} shared URLs, "
              f"containment {r.norm_containment:.3f}")
    print("\nEach flagged pair counts as ONE corpus. The smaller member is dropped")
    print("in Cell 6. Report the overlap - two datasets circulating under different")
    print("names and cited as independent is a finding in its own right.")
else:
    print("\nNo pair exceeds 0.5 containment - the corpora are distinct.")

        corpus_a          corpus_b  norm_overlap  norm_jaccard  norm_containment  domain_overlap  domain_jaccard  domain_containment
       hannousse  kaggle_linkphish         11070        1.0000            1.0000            6635          1.0000              1.0000
kaggle_malicious mendeley_phishurl          4618        0.0402            0.0773            5516          0.1155              0.2097
kaggle_malicious  kaggle_linkphish           347        0.0049            0.0313             769          0.0239              0.1159
       hannousse  kaggle_malicious           347        0.0049            0.0313             769          0.0239              0.1159
       hannousse mendeley_phishurl           153        0.0022            0.0138             695          0.0211              0.1047
kaggle_linkphish mendeley_phishurl           153        0.0022            0.0138             695          0.0211              0.1047
        phiusiil  kaggle_linkphish            62        0.0009       

## Cell 6 — Canonical feature matrices

In [8]:
DROP = set()
if len(flagged):
    for _, r in flagged.iterrows():
        smaller = (r.corpus_a if len(CORPORA[r.corpus_a]) < len(CORPORA[r.corpus_b])
                   else r.corpus_b)
        DROP.add(smaller)
    print(f"dropping as non-independent: {sorted(DROP)}\n")

canon = {}
mal_rows = []
for key, df in CORPORA.items():
    if key in DROP:
        continue
    urls = df["url"].astype(str)
    mal = urls.map(uf.is_malformed)
    X = uf.build_canonical_matrix(urls)
    dom = urls.map(uf.registrable_domain)
    canon[key] = {"X": X.reset_index(drop=True),
                  "y": df["label"].reset_index(drop=True),
                  "domain": dom.reset_index(drop=True)}
    mal_rows.append({"corpus": key, "n": len(df), "malformed_n": int(mal.sum()),
                     "malformed_pct": round(100 * mal.mean(), 4),
                     "unique_domains": int(dom.nunique())})
    print(f"{key:20s} X={str(X.shape):>13s}  domains={dom.nunique():>7,}  "
          f"malformed={mal.sum():>5,} ({100*mal.mean():.3f}%)")

pd.DataFrame(mal_rows).to_csv(
    REPO_DIR / "reports" / "tables" / "data_quality.csv", index=False)
print(f"\n{len(canon)} corpora carried forward -> reports/tables/data_quality.csv")

dropping as non-independent: ['hannousse']

phiusiil             X=  (60000, 32)  domains= 46,716  malformed=    0 (0.000%)
kaggle_malicious     X=  (60000, 32)  domains= 26,307  malformed=    4 (0.007%)
kaggle_linkphish     X=  (19431, 32)  domains=  6,635  malformed=    0 (0.000%)
mendeley_phishurl    X=  (60000, 32)  domains= 26,949  malformed=    0 (0.000%)

4 corpora carried forward -> reports/tables/data_quality.csv


## Cell 7 — Full transfer matrix, multiple seeds

Every ordered pair of corpora, five seeds. Diagonal entries use domain-grouped
splits within the corpus; off-diagonal entries train on all of one and test on all
of the other.

Domain-grouped splitting can occasionally place a single class on one side of the
boundary in a small or highly clustered corpus. Those partitions produce an
undefined AUC, so they are skipped with a printed warning rather than silently
becoming NaN and contaminating the aggregate. If you see many skips for one
corpus, its domain structure is too concentrated for grouped splitting and you
should report that instead of working around it.

In [9]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, f1_score, matthews_corrcoef

SEEDS = (42, 43, 44, 45, 46)
KEYS = list(canon)

def make_model(seed):
    return XGBClassifier(n_estimators=400, max_depth=7, learning_rate=0.1,
                         subsample=0.9, colsample_bytree=0.9, n_jobs=-1,
                         eval_metric="logloss", random_state=seed, verbosity=0)

records = []
raw_scores = {}     # (train, test, seed) -> (y_true, scores)

for src in KEYS:
    for dst in KEYS:
        for seed in SEEDS:
            Xs, ys = canon[src]["X"], canon[src]["y"]
            if src == dst:
                tr, te = uf.grouped_split(Xs, ys, canon[src]["domain"], seed=seed)
                Xtr, ytr = Xs.iloc[tr], ys.iloc[tr]
                Xte, yte = Xs.iloc[te], ys.iloc[te]
            else:
                Xd, yd = canon[dst]["X"], canon[dst]["y"]
                cols = [col for col in Xs.columns if col in Xd.columns]
                Xtr, ytr, Xte, yte = Xs[cols], ys, Xd[cols], yd
            yte_arr = np.asarray(yte)
            if len(np.unique(yte_arr)) < 2 or len(np.unique(np.asarray(ytr))) < 2:
                print(f"  [skip] {src}->{dst} seed={seed}: single-class partition")
                continue
            m = make_model(seed).fit(Xtr, ytr)
            s = m.predict_proba(Xte)[:, 1]
            raw_scores[(src, dst, seed)] = (yte_arr, s)
            records.append({
                "train": src, "test": dst, "seed": seed,
                "kind": "within" if src == dst else "cross",
                "roc_auc": roc_auc_score(yte, s),
                "f1": f1_score(yte, (s >= 0.5).astype(int), zero_division=0),
                "mcc": matthews_corrcoef(yte, (s >= 0.5).astype(int)),
            })
        print(f"  {src} -> {dst}: done")

transfer = pd.DataFrame(records)
transfer.to_csv(REPO_DIR / "reports" / "tables" / "transfer_matrix.csv", index=False)

piv = transfer.pivot_table(index="train", columns="test", values="roc_auc").round(4)
print("\nMean ROC-AUC over seeds (rows = train, cols = test):")
print(piv.to_string())

  phiusiil -> phiusiil: done
  phiusiil -> kaggle_malicious: done
  phiusiil -> kaggle_linkphish: done
  phiusiil -> mendeley_phishurl: done
  kaggle_malicious -> phiusiil: done
  kaggle_malicious -> kaggle_malicious: done
  kaggle_malicious -> kaggle_linkphish: done
  kaggle_malicious -> mendeley_phishurl: done
  kaggle_linkphish -> phiusiil: done
  kaggle_linkphish -> kaggle_malicious: done
  kaggle_linkphish -> kaggle_linkphish: done
  kaggle_linkphish -> mendeley_phishurl: done
  mendeley_phishurl -> phiusiil: done
  mendeley_phishurl -> kaggle_malicious: done
  mendeley_phishurl -> kaggle_linkphish: done
  mendeley_phishurl -> mendeley_phishurl: done

Mean ROC-AUC over seeds (rows = train, cols = test):
test               kaggle_linkphish  kaggle_malicious  mendeley_phishurl  phiusiil
train                                                                             
kaggle_linkphish             0.9416            0.5395             0.9390    0.8691
kaggle_malicious             0.44

## Cell 8 — Confidence intervals

Stratified bootstrap on the test set, 2,000 resamples, plus the DeLong analytic
interval as a cross-check. The two should agree closely; if they do not, treat the
bootstrap as authoritative and investigate the class balance.

In [10]:
import src.evaluation.stats as ST
importlib.reload(ST)

ci_rows = []
for (src, dst, seed), (y, s) in raw_scores.items():
    if seed != SEEDS[0]:
        continue
    pt, lo, hi = ST.bootstrap_ci(y, s, roc_auc_score, n_boot=2000, seed=seed)
    d_auc, d_lo, d_hi = ST.delong_ci(y, s)
    ci_rows.append({
        "train": src, "test": dst,
        "kind": "within" if src == dst else "cross",
        "roc_auc": round(pt, 4),
        "boot_lo": round(lo, 4), "boot_hi": round(hi, 4),
        "delong_lo": round(d_lo, 4), "delong_hi": round(d_hi, 4),
        "ci_width": round(hi - lo, 4),
    })

cis = pd.DataFrame(ci_rows).sort_values(["kind", "roc_auc"])
print(cis.to_string(index=False))
cis.to_csv(REPO_DIR / "reports" / "tables" / "transfer_cis.csv", index=False)

            train              test   kind  roc_auc  boot_lo  boot_hi  delong_lo  delong_hi  ci_width
 kaggle_malicious mendeley_phishurl  cross   0.1695   0.1656   0.1736     0.1654     0.1737    0.0081
         phiusiil  kaggle_malicious  cross   0.2346   0.2290   0.2403     0.2291     0.2402    0.0113
mendeley_phishurl  kaggle_malicious  cross   0.2367   0.2313   0.2421     0.2312     0.2421    0.0108
 kaggle_malicious          phiusiil  cross   0.4071   0.4021   0.4124     0.4019     0.4124    0.0104
 kaggle_malicious  kaggle_linkphish  cross   0.4470   0.4389   0.4546     0.4389     0.4552    0.0158
 kaggle_linkphish  kaggle_malicious  cross   0.5383   0.5322   0.5440     0.5324     0.5442    0.0118
         phiusiil  kaggle_linkphish  cross   0.6946   0.6876   0.7021     0.6873     0.7019    0.0145
mendeley_phishurl  kaggle_linkphish  cross   0.7880   0.7817   0.7943     0.7817     0.7942    0.0126
 kaggle_linkphish          phiusiil  cross   0.8778   0.8750   0.8810     0.8748  

## Cell 9 — Within versus cross, with a permutation test

The paper's central quantitative claim. We test whether the observed gap could
arise by chance reassignment of the transfer results, and report Cliff's delta as
a distribution-free effect size.

In [11]:
within = transfer.loc[transfer.kind == "within", "roc_auc"].values
cross  = transfer.loc[transfer.kind == "cross",  "roc_auc"].values

n_bad = int(np.isnan(within).sum() + np.isnan(cross).sum())
within = within[~np.isnan(within)]
cross  = cross[~np.isnan(cross)]
if n_bad:
    print(f"NOTE: {n_bad} undefined AUC value(s) dropped (degenerate partitions).")
if len(within) < 2 or len(cross) < 2:
    raise RuntimeError("Too few valid results for the permutation test. "
                       "Check Cell 7 for skipped partitions.")

obs, p_perm = ST.permutation_gap_test(within, cross, n_perm=10000)
delta = ST.cliffs_delta(within, cross)

print(f"within  n={len(within):3d}  mean={within.mean():.4f}  sd={within.std(ddof=1):.4f}")
print(f"cross   n={len(cross):3d}  mean={cross.mean():.4f}  sd={cross.std(ddof=1):.4f}")
print(f"\nobserved gap      : {obs:.4f}")
print(f"permutation p     : {p_perm:.5f}  (10,000 permutations)")
print(f"Cliff's delta     : {delta:.4f}   (|d| > 0.474 is a large effect)")
print(f"\nNote: the smallest attainable p is {1/(10000+1):.5f}.")
print("Report it as p < 0.001 rather than as an exact value.")

within  n= 20  mean=0.9713  sd=0.0277
cross   n= 60  mean=0.6018  sd=0.2895

observed gap      : 0.3695
permutation p     : 0.00010  (10,000 permutations)
Cliff's delta     : 0.8800   (|d| > 0.474 is a large effect)

Note: the smallest attainable p is 0.00010.
Report it as p < 0.001 rather than as an exact value.


## Cell 10 — Asymmetry test

The paper's key inferential step. If poor transfer were a general property of
URL-lexical features, transfer in both directions between a pair of corpora would
degrade similarly. Directional asymmetry instead localises the problem to a
particular corpus.

DeLong compares the two directions where they share a test set, and Holm--Bonferroni
controls the family-wise error rate across all pairs.

In [12]:
pairs, pvals = [], []
for i, a in enumerate(KEYS):
    for b in KEYS[i+1:]:
        # both directions evaluated on corpus b's data:
        #   a -> b   (transfer)   vs   b -> b   (within, same test partition is
        #   not identical, so we compare distributions rather than paired scores)
        ab = transfer[(transfer.train == a) & (transfer.test == b)]["roc_auc"].values
        ba = transfer[(transfer.train == b) & (transfer.test == a)]["roc_auc"].values
        if len(ab) == 0 or len(ba) == 0:
            continue
        from scipy import stats as sps
        try:
            p = float(sps.mannwhitneyu(ab, ba, alternative="two-sided").pvalue)
        except Exception:
            p = np.nan
        pairs.append({"corpus_a": a, "corpus_b": b,
                      "auc_a_to_b": round(ab.mean(), 4),
                      "auc_b_to_a": round(ba.mean(), 4),
                      "asymmetry": round(abs(ab.mean() - ba.mean()), 4),
                      "p_raw": p})
        pvals.append(p)

asym = pd.DataFrame(pairs)
if len(asym):
    adj, rej = ST.holm_bonferroni(asym["p_raw"].values, alpha=0.05)
    asym["p_holm"] = np.round(adj, 5)
    asym["significant"] = rej
    asym = asym.sort_values("asymmetry", ascending=False)
    print(asym.to_string(index=False))
    asym.to_csv(REPO_DIR / "reports" / "tables" / "asymmetry.csv", index=False)
    print(f"\n{int(rej.sum())} of {len(asym)} corpus pairs show significant "
          f"directional asymmetry after Holm correction.")
else:
    print("Need at least two corpora for the asymmetry test.")

        corpus_a          corpus_b  auc_a_to_b  auc_b_to_a  asymmetry    p_raw  p_holm  significant
        phiusiil  kaggle_linkphish      0.6778      0.8691     0.1913 0.007937 0.04762         True
        phiusiil  kaggle_malicious      0.2258      0.3940     0.1682 0.007937 0.04762         True
kaggle_linkphish mendeley_phishurl      0.9390      0.7880     0.1509 0.007937 0.04762         True
kaggle_malicious  kaggle_linkphish      0.4496      0.5395     0.0899 0.007937 0.04762         True
        phiusiil mendeley_phishurl      0.9307      0.9790     0.0483 0.007937 0.04762         True
kaggle_malicious mendeley_phishurl      0.1932      0.2361     0.0429 0.007937 0.04762         True

6 of 6 corpus pairs show significant directional asymmetry after Holm correction.


**Interpreting asymmetry.** A large, significant asymmetry for a pair means one
of the two corpora is a much worse *source* of training data than the other is,
which is what a structural artefact produces. Cross-reference every significant
pair against the `degenerate_cells` column from Cell 5. Where the two agree, the
structural audit explains the transfer behaviour, and that convergence of two
independent diagnostics is the strongest evidence the paper can offer.

Where they disagree, say so. A significant asymmetry with a clean structural
profile means something else is going on, and it should be reported as an open
question rather than smoothed over.

## Cell 11 — Model-level comparison with DeLong

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

COMPARE = {
    "NaiveBayes":  lambda s: make_pipeline(StandardScaler(), GaussianNB()),
    "LogisticReg": lambda s: make_pipeline(StandardScaler(),
                       LogisticRegression(max_iter=2000, random_state=s)),
    "RandomForest": lambda s: RandomForestClassifier(n_estimators=300, n_jobs=-1,
                                                     random_state=s),
    "XGBoost": make_model,
}

rows, pv = [], []
for key in KEYS:
    X, y, dom = canon[key]["X"], canon[key]["y"], canon[key]["domain"]
    tr, te = uf.grouped_split(X, y, dom, seed=42)
    yte = np.asarray(y.iloc[te])
    sc = {}
    for name, f in COMPARE.items():
        m = f(42).fit(X.iloc[tr], y.iloc[tr])
        sc[name] = m.predict_proba(X.iloc[te])[:, 1]
    for name in COMPARE:
        if name == "XGBoost":
            continue
        a, b, z, p = ST.delong_test(yte, sc["XGBoost"], sc[name])
        rows.append({"corpus": key, "baseline": "XGBoost", "model": name,
                     "auc_xgb": round(a, 4), "auc_other": round(b, 4),
                     "diff": round(a - b, 4), "p_raw": p})
        pv.append(p)

cmp = pd.DataFrame(rows)
if len(cmp):
    adj, rej = ST.holm_bonferroni(cmp["p_raw"].values)
    cmp["p_holm"] = np.round(adj, 6)
    cmp["significant"] = rej
    print(cmp.to_string(index=False))
    cmp.to_csv(REPO_DIR / "reports" / "tables" / "delong_models.csv", index=False)
    print("\nWatch the `diff` column against significance: on a corpus with a")
    print("structural artefact, XGBoost may beat naive Bayes significantly while")
    print("the difference is a few thousandths of AUC and operationally meaningless.")

           corpus baseline        model  auc_xgb  auc_other    diff         p_raw   p_holm  significant
         phiusiil  XGBoost   NaiveBayes   0.9991     0.9977  0.0014  2.451659e-05 0.000147         True
         phiusiil  XGBoost  LogisticReg   0.9991     0.9985  0.0007  2.735342e-02 0.109414        False
         phiusiil  XGBoost RandomForest   0.9991     0.9994 -0.0002  2.072654e-01 0.414531        False
 kaggle_malicious  XGBoost   NaiveBayes   0.9440     0.7959  0.1481 3.708407e-169 0.000000         True
 kaggle_malicious  XGBoost  LogisticReg   0.9440     0.8445  0.0995 2.150541e-124 0.000000         True
 kaggle_malicious  XGBoost RandomForest   0.9440     0.9474 -0.0033  3.694469e-02 0.110834        False
 kaggle_linkphish  XGBoost   NaiveBayes   0.9477     0.8240  0.1236  2.837138e-81 0.000000         True
 kaggle_linkphish  XGBoost  LogisticReg   0.9477     0.9037  0.0440  3.302753e-30 0.000000         True
 kaggle_linkphish  XGBoost RandomForest   0.9477     0.9406  0.0

## Cell 12 — Checkpoint and commit

In [14]:
ck6 = Checkpoint(run_id="extended_corpora")
ck6.save("transfer_matrix", transfer, params={"seeds": list(SEEDS), "corpora": KEYS})
ck6.save("structural_audit", audit, params={"version": 1})
ck6.save("confidence_intervals", cis, params={"n_boot": 2000})
print(ck6.summary().to_string())

[ckpt] saved transfer_matrix__41238fb894.joblib  (0.003 MB)
[ckpt] saved structural_audit__fce8ed52ad.joblib  (0.001 MB)
[ckpt] saved confidence_intervals__d57c6e35a4.joblib  (0.001 MB)
                                                        stage             saved_at size_mb git_commit
transfer_matrix__05777665ef.joblib            transfer_matrix  2026-07-24 08:38:11   0.001      nogit
structural_audit__fce8ed52ad.joblib          structural_audit  2026-07-24 08:54:13   0.001      nogit
confidence_intervals__d57c6e35a4.joblib  confidence_intervals  2026-07-24 08:54:14   0.001      nogit
transfer_matrix__41238fb894.joblib            transfer_matrix  2026-07-24 08:54:13   0.003      nogit


In [15]:
import shutil, glob, subprocess
for folder in ["src", "reports"]:
    dst = DRIVE_DIR / "_repo_backup" / folder
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(REPO_DIR / folder, dst)
subprocess.run(["git", "-C", str(REPO_DIR), "add", "-A"])
r = subprocess.run(["git", "-C", str(REPO_DIR), "commit", "-m",
                    "feat: additional corpora, bootstrap CIs, DeLong, Holm correction"],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)

fatal: not a git repository (or any of the parent directories): .git



---
### New outputs

| File | Use |
|---|---|
| `structural_audit.csv` | New Table 2 — the survey across all corpora |
| `corpus_overlap.csv` | Table 3 — corpus independence check |
| `data_quality.csv` | Malformed-URL rates, for the Data section |
| `transfer_matrix.csv` | Expanded Table 4, now $N \times N$ with seeds |
| `transfer_cis.csv` | Confidence intervals for every transfer cell |
| `asymmetry.csv` | The directional test, Holm-corrected |
| `delong_models.csv` | Model comparison with paired significance |

### What to report, and how

Give the permutation test as $p < 0.001$ rather than an exact figure; with 10,000
permutations that is the resolution limit and quoting more precision is false.

Report Cliff's delta next to it. The gap is large enough that the effect size is
the informative number and the p-value is close to a formality.

State the corpus count explicitly in the abstract. "We audit $N$ public corpora"
is the claim the added datasets buy you, and it is worth more than any additional
decimal place on a p-value.

If some downloads failed, report the corpora you actually analysed rather than the
ones you attempted. Cell 4's summary table is the authoritative list.